In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler


# 1. LOAD DATA

df = pd.read_csv("raw_loan_dataset.csv")

print("Initial data:")
print(df.head())
print(df.info())
print(df.isnull().sum())


# 2. CLEAN CURRENCY ($, commas)

df["Income"] = df["Income"].astype(str).str.replace("$", "", regex=False).str.replace(",", "")
df["LoanAmount"] = df["LoanAmount"].astype(str).str.replace("$", "", regex=False).str.replace(",", "")

df["Income"] = pd.to_numeric(df["Income"], errors="coerce")
df["LoanAmount"] = pd.to_numeric(df["LoanAmount"], errors="coerce")


# 3. CLEAN YES/NO VALUES

def clean_yes_no(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()

    if x in ["yes", "y", "yse", "approved", "yes "]:
        return "Yes"
    if x in ["no", "n", "rejected", "no "]:
        return "No"
    return np.nan

for col in ["HasCollateral", "PreviousDefaults", "Approved"]:
    df[col] = df[col].apply(clean_yes_no)
    print(f"\n{col} value counts:")
    print(df[col].value_counts())


# 4. IMPUTE MISSING VALUES

num_cols = ["Income", "CreditScore", "EmploymentYears", "LoanAmount"]

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in ["HasCollateral", "PreviousDefaults", "Approved"]:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing values after imputation:")
print(df.isnull().sum())


# 5. REMOVE DUPLICATES

print("\nBefore duplicates:", df.shape)
df = df.drop_duplicates()
print("After duplicates:", df.shape)


# 6. OUTLIER CAPPING (IQR)

cols = ["Income", "CreditScore", "LoanAmount", "EmploymentYears"]

for col in cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)


# 7. ENCODING

mapping = {"No": 0, "Yes": 1}

df["Approved"] = df["Approved"].map(mapping)
df["HasCollateral"] = df["HasCollateral"].map(mapping)
df["PreviousDefaults"] = df["PreviousDefaults"].map(mapping)

print("\nApproved distribution:")
print(df["Approved"].value_counts())

# 8. FEATURE ENGINEERING

df["DebtToIncome"] = df["LoanAmount"] / df["Income"]
df["IncomePerYearEmployed"] = df["Income"] / df["EmploymentYears"]


# 9. SCALING (RobustScaler)

scaler = RobustScaler()

scale_cols = ["Income", "CreditScore", "LoanAmount",
              "EmploymentYears", "DebtToIncome",
              "IncomePerYearEmployed"]

df[scale_cols] = scaler.fit_transform(df[scale_cols])


# 10. FINAL CHECKS

print("\nFinal info:")
print(df.info())
print(df.isnull().sum())
print(df.head())


# 11. SAVE CLEAN DATASET

df.to_csv("clean_loan_dataset.csv", index=False)

print("\nSaved clean_loan_dataset.csv successfully!")

Initial data:
    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  
<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     str    
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     str    
 4   HasCo